In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = 2  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    "4b": ["lpips==0.1.4", "pycolmap==4.2.1", "ninja"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1", "open3d==0.20.0"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {"4b": ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"],
                      6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}
# Reference objects with a course-made TRELLIS.2 archive, generated-<object>.tar.gz.
GENERATED_OBJECTS = ("boxed_beverage_001", "hammer_014")
# The TRELLIS.2 model revision those archives were generated with (model_revision in trellis2.json).
GENERATED_MODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def prev_folder(stage):
    """The folder a stage reads: capture/ for stage 1, stage4/ for stage 4b, stage<N-1>/ otherwise.

    Stage 5 reads stage4/ too: stage 4b refines the maps but leaves the mesh to stage 4.
    """
    if stage == 1:
        return "capture"
    if stage == "4b":
        return "stage4"
    return f"stage{stage - 1}"


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (prev_folder(stage)), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    if not (PROJECT / prev_folder(stage) / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None, dest=None):
    """Download a reference archive and move its top-level folders (or only `names`) into dest.

    dest defaults to PROJECT. A non-empty folder of yours that a reference folder replaces is
    kept as <name>.yours (an older .yours is replaced); ground_truth/ is replaced without a copy.
    Nothing moves until the archive is fully extracted.
    """
    dest = PROJECT if dest is None else Path(dest)
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = dest / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = dest / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = dest / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = dest / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"{prev_folder(stage)}.tar.gz"
    print(f"{prev_folder(stage)}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = prev_folder(stage)
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = "stage0.tar.gz" if stage == 1 else f"{prev}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


def ensure_generated(obj):
    """Fetch the course's TRELLIS.2 archive for a reference object and return its folder.

    generated-<obj>.tar.gz holds that object's stage folders, ground_truth/ and trellis2/. They
    go to generated/<obj>/, which is replaced whole on each download, so your own stage folders
    are never touched. A failed download leaves the previous generated/<obj>/ as it was. When
    generated/<obj>/trellis2/trellis2.json is already there with GENERATED_MODEL_REVISION, the
    folder is kept and nothing is downloaded.
    """
    if obj not in GENERATED_OBJECTS:
        raise ValueError(f"no generated archive for {obj!r}; the course ships {', '.join(GENERATED_OBJECTS)}")
    out = PROJECT / "generated" / obj
    try:
        revision = json.loads((out / "trellis2" / "trellis2.json").read_text()).get("model_revision")
    except (OSError, ValueError, AttributeError):
        revision = None
    if revision == GENERATED_MODEL_REVISION:
        print(f"generated/{obj}/ is already here (TRELLIS.2 revision {revision[:7]}), so it is not downloaded again.")
        return out
    with atomic_dir(out) as tmp:
        _extract_reference(f"generated-{obj}.tar.gz", dest=tmp)
    return out


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you separate your object from the table, the board and the room behind it, in every frame. Stage 3 builds geometry only from the pixels these masks keep, so a mask that slips onto the table here becomes a lump of table on your mesh there.

You prompt a promptable video segmenter once, with a box around the object in the first frame, and it follows the object through the rest of the orbit. The model is SAM 2 (Ravi et al., arXiv 2408.00714), which extends the image segmenter of Segment Anything [@Kirillov2023-gi] to video by adding a memory of the frames it has already segmented. You use the SAM 2.1 checkpoint `sam2.1_hiera_base_plus`, released under the Apache-2.0 license. At the end of the section you train two quick Gaussian splats [@Kerbl2023-rg], one with the masks and one without, using the gsplat implementation (Apache-2.0) [@Ye2024-xk]. They are a first look at your object in 3D and a check that the capture is good enough to continue.

Runtime budget on a Colab T4: about 6 minutes for 100 frames once the session is set up. The first run in a new session takes about 20 minutes, of which about 14 minutes is one-time setup.


In [ ]:
import json
import shutil
import time

import numpy as np

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
STAGE1 = PROJECT / "stage1"
REFERENCE = on_reference(STAGE)  # the reference stage1/ carries a .reference marker
WORK = (Path("/content/work") if ON_COLAB else PROJECT / ".work") / "stage2"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
frames = sorted((STAGE1 / "frames").glob("*.jpg"))
heldout = set((STAGE1 / "heldout.txt").read_text().split())
train = [p for p in frames if p.name not in heldout]
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print(f"{len(frames)} frames from stage 1: {len(train)} for training, {len(heldout)} held out")

## The box prompt

The first frame is drawn below with a pixel grid. Read off the top-left corner $(x_0, y_0)$ and the bottom-right corner $(x_1, y_1)$ of a box that holds the whole object and as little else as possible, and enter them as `BOX_XYXY`. The box tells the model which object you mean. A single click on the reference carton selects only its printed label, which is a plausible object on its own, while a box around the whole carton selects the carton.


In [ ]:
import io

import cv2
import matplotlib.pyplot as plt
from IPython.display import Image, display

first = cv2.cvtColor(cv2.imread(str(frames[0])), cv2.COLOR_BGR2RGB)
h, w = first.shape[:2]
fig, ax = plt.subplots(figsize=(13, 13 * h / w))
ax.imshow(first)
ax.set_xticks(range(0, w + 1, 100))
ax.set_yticks(range(0, h + 1, 100))
ax.tick_params(labelsize=7)
plt.setp(ax.get_xticklabels(), rotation=90)
ax.grid(color="yellow", linewidth=0.5, alpha=0.6)
ax.set_title(f"{frames[0].name}, {w} x {h} pixels: read the box corners off the grid")
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=90, pil_kwargs={"quality": 85})  # photos: JPEG keeps the page small
plt.close(fig)
display(Image(data=buf.getvalue()))

In [ ]:
import cv2

BOX_XYXY = None  # your box as (x0, y0, x1, y1) in pixels: top-left and bottom-right corners of the object

h, w = cv2.imread(str(frames[0])).shape[:2]
box = BOX_XYXY
if box is None and REFERENCE:  # the reference object: the tight box around the dataset's matting of this frame
    ys, xs = np.nonzero(cv2.imread(str(PROJECT / "ground_truth" / "matting" / f"{frames[0].stem}.png"),
                                   cv2.IMREAD_GRAYSCALE) > 127)
    box = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
assert box is not None, "Set BOX_XYXY to the box around your object in the first frame."
x0, y0, x1, y1 = box
assert 0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h, f"box {box} is not inside the {w} x {h} frame"
print(f"box prompt on {frames[0].name}: {box}, {x1 - x0} x {y1 - y0} pixels")

## Tracking with a memory bank

SAM 2 processes the frames in order. An image encoder turns frame $t$ into a grid of features $F_t$. Before the mask decoder sees them, a memory attention block lets $F_t$ attend to a memory bank

$$
\mathcal{M}_t = \{ m_0 \} \cup \{ m_{t-1}, m_{t-2}, \dots, m_{t-6} \},
$$

where each memory $m_s$ is a compact encoding of frame $s$ fused with the mask predicted for it. The prompted frame $m_0$ stays in the bank for the whole video, and the six most recent frames slide along with $t$. The decoder then predicts the mask of frame $t$ from the conditioned features, and that mask is encoded into $m_t$ for the frames after it. The bank also keeps a short vector per frame, an object pointer, that summarizes what the object looked like.

The prompted frame fixes which object you meant, and the recent frames carry its current appearance. That lets the tracker follow a carton whose visible face, size and lighting change as you walk around it. Nothing in the bank knows about 3D. When the recent memories go wrong, for instance after the mask jumped to the board, the error can carry forward until the prompted frame pulls it back.

Write `track_masks`. It returns one boolean mask per frame, for every frame of stage 1, held-out frames included: stage 6 needs their masks to score your twin on photos it never trained on.


In [ ]:
import numpy as np
import torch
from pathlib import Path


def track_masks(frames: list[Path], box_xyxy: tuple[int, int, int, int]) -> dict[str, np.ndarray]:
    """Segment the object in every frame with the SAM 2.1 video predictor.

    1. Download the checkpoint sam2.1_hiera_base_plus.pt from the Hugging Face repo
       facebook/sam2.1-hiera-base-plus, and build the predictor with
       sam2.build_sam.build_sam2_video_predictor("configs/sam2.1/sam2.1_hiera_b+.yaml", ...).
    2. The predictor reads a folder of JPEG frames named 00000.jpg, 00001.jpg, ... in video
       order. Link or copy the frames into a temporary folder under those names.
    3. Under torch.inference_mode() and torch.autocast("cuda", dtype=torch.float16), call
       init_state(folder, offload_video_to_cpu=True), add the box on frame 0 with
       add_new_points_or_box, then iterate propagate_in_video. A pixel belongs to the
       object where the returned logit is above 0.

    Return {frame name: boolean mask with the frame's height and width}, for every frame.
    """
    raise NotImplementedError

In [ ]:
t = time.time()
masks = track_masks(frames, box)
assert sorted(masks) == [p.name for p in frames], "track_masks must return one mask per frame"
print(f"{len(masks)} masks in {time.time() - t:.0f} s")

## The drift check

The tracker gives no warning when it loses the object, so you check its output against a simple expectation. As the camera moves smoothly, the object's area in the image changes smoothly too. Let $a_i$ be the mask area of frame $i$ in pixels and

$$
\bar a_i = \operatorname{median}(a_{i-1}, a_{i+1}).
$$

Frame $i$ is flagged when $|a_i - \bar a_i| > 0.3\, \bar a_i$, that is, when its area is more than 30% away from what its neighbors suggest. An empty mask is always flagged. A flagged frame is not used as a neighbor, so a single lost frame does not also flag the two good frames around it. Flagged frames are listed below, kept out of both quick splats and recorded in `stage2/drift.json` for later stages.

The rule sees only the area. A mask that slides onto a neighboring object of about the same size passes it.


In [ ]:
import numpy as np


def flag_drift(areas, threshold: float = 0.3) -> list[int]:
    """Indices of frames whose mask area is more than `threshold` away from its neighbors.

    The reference for frame i is the median of the areas of its two neighbors, i - 1 and
    i + 1 (one neighbor at either end of the sequence). An empty mask is always flagged.
    A flagged frame is not used as a neighbor: the frames on either side of it are compared
    with the next unflagged frame instead, so one lost frame does not also flag the two
    good frames around it. Frames are flagged one at a time, the furthest from its
    reference first.
    """
    a = np.asarray(areas, float)
    n = len(a)
    flagged = {i for i in range(n) if a[i] <= 0}

    def neighbors(i):
        out = []
        for step in (-1, 1):
            j = i + step
            while 0 <= j < n and j in flagged:
                j += step
            if 0 <= j < n:
                out.append(a[j])
        return out

    while True:
        worst, worst_gap = None, 0.0
        for i in range(n):
            if i in flagged:
                continue
            near = neighbors(i)
            if not near:
                continue
            ref = float(np.median(near))
            gap = abs(np.log(a[i] / ref))  # symmetric: a 30% jump up and down rank alike
            if abs(a[i] - ref) > threshold * ref and gap > worst_gap:
                worst, worst_gap = i, gap
        if worst is None:
            return sorted(flagged)
        flagged.add(worst)

In [ ]:
areas = [int(masks[p.name].sum()) for p in frames]
flagged = flag_drift(areas)
print(f"mask area per frame: min {min(areas):,}, median {int(np.median(areas)):,}, max {max(areas):,} pixels "
      f"(max / min = {max(areas) / max(min(areas), 1):.2f})")
step = np.abs(np.diff(areas)) / np.maximum(np.minimum(areas[:-1], areas[1:]), 1)
j = int(step.argmax())
print(f"largest change between consecutive frames: {100 * step[j]:.1f}% ({frames[j].name} to {frames[j + 1].name})")
if flagged:
    print(f"warning: {len(flagged)} frame(s) differ by more than 30% from their neighbors:")
    for i in flagged:
        print(f"  {frames[i].name}: {areas[i]:,} pixels")
else:
    print("no frame differs by more than 30% from its neighbors")

In [ ]:
import io

import cv2
import matplotlib.pyplot as plt
from IPython.display import Image, display


def overlay(p, mask, color=(255, 80, 0)):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    img[mask] = (0.5 * img[mask] + 0.5 * np.array(color)).astype(np.uint8)
    return img


fig = plt.figure(figsize=(14, 7.5))
grid = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.1])
show = frames[:: max(1, len(frames) // 8)][:8]
for k, p in enumerate(show):
    ax = fig.add_subplot(grid[k // 4, k % 4])
    ax.imshow(overlay(p, masks[p.name]))
    ax.set_title(p.name, fontsize=9)
    ax.axis("off")
ax = fig.add_subplot(grid[2, :])
ax.plot(range(len(areas)), areas, ".-", color="tab:blue", lw=1, ms=3)
if flagged:
    ax.plot(flagged, [areas[i] for i in flagged], "o", mfc="none", mec="tab:red", ms=9, label="flagged")
    ax.legend()
ax.set_xlabel("frame (stage 1 order)")
ax.set_ylabel("mask area (pixels)")
ax.grid(alpha=0.3)
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=80, pil_kwargs={"quality": 85})
plt.close(fig)
display(Image(data=buf.getvalue()))

## A quick splat with and without masks

A 3D Gaussian splat represents the scene as a set of Gaussians, each with a center $\mu_k$, a covariance $\Sigma_k$, an opacity $o_k$ and a color $c_k$. To render a pixel, the Gaussians are projected into the image and blended front to back,

$$
C = \sum_k c_k \alpha_k \prod_{j<k} (1 - \alpha_j) + T\, b, \qquad T = \prod_k (1 - \alpha_k),
$$

where $\alpha_k$ is the opacity of Gaussian $k$ weighted by its projected 2D footprint at that pixel, and $b$ is the background color seen through what remains, with transmittance $T$. Training starts from one Gaussian per sparse point of stage 1, adjusts all parameters to match the photos with the loss $0.8\, L_1 + 0.2\,(1 - \mathrm{SSIM})$, and splits or clones Gaussians where the image gradient says detail is missing.

Without masks, every pixel of every photo is a target, so the splat spends its Gaussians on whatever the photos show: the room, the floor, the board. With masks, every pixel outside the mask is replaced by a background color drawn at random at each step, and the same color is used for $b$. The only way to match a background that keeps changing is $T = 1$, no Gaussian at all, so the splat outside the object empties out.

Both splats train for 2,000 steps at half resolution on the training frames. Each is then measured three ways: its number of Gaussians, the share of Gaussian centers inside the object's visual hull (centers that project inside the mask in at least 90% of the training views that see them), and the PSNR over the object's pixels in the held-out frames. The third row cuts the unmasked splat to its Gaussians inside the visual hull, which is what you would get by trying to remove the background after training instead of before.


In [ ]:
import contextlib
import io

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from scipy.spatial import cKDTree


def load_views(stage1: Path, frames, masks=None, data_factor: int = 2):
    """Undistorted images (and masks) at 1/data_factor resolution, with their cameras.

    stage1/sparse/0 is already in meters in the world frame, so the splat is too.
    """
    from pycolmap import SceneManager

    with contextlib.redirect_stdout(io.StringIO()):  # the reader prints a note about the image folder
        model = SceneManager(str(Path(stage1) / "sparse" / "0"))
    model.load_cameras()
    model.load_images()
    model.load_points3D()
    by_name = {im.name: im for im in model.images.values()}
    views, skipped = [], []
    for p in frames:
        im = by_name.get(Path(p).name)
        if im is None:
            skipped.append(Path(p).name)
            continue
        cam = model.cameras[im.camera_id]
        K = np.array([[cam.fx, 0, cam.cx], [0, cam.fy, cam.cy], [0, 0, 1.0]])
        K[:2] /= data_factor
        dist = np.array([getattr(cam, k, 0.0) for k in ("k1", "k2", "p1", "p2")])
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        size = (img.shape[1] // data_factor, img.shape[0] // data_factor)
        img = cv2.undistort(cv2.resize(img, size, interpolation=cv2.INTER_AREA), K, dist)
        view = {"name": Path(p).name, "K": K, "image": img}
        if masks is not None:
            m = cv2.resize(masks[Path(p).name].astype(np.float32), size, interpolation=cv2.INTER_AREA)
            view["mask"] = cv2.undistort(m, K, dist) > 0.5
        viewmat = np.eye(4)
        viewmat[:3, :3], viewmat[:3, 3] = im.R(), im.tvec
        view["viewmat"] = viewmat
        views.append(view)
    if skipped:  # write_stage1 accepts a model with 80% of the frames registered
        print(f"{len(skipped)} of {len(frames)} frames have no pose in stage1/sparse/0 (stage 1 did not register "
              f"them) and are left out: {', '.join(skipped)}")
    return views, model.points3D, model.point3D_colors


def _ssim(x, y, size=11, sigma=1.5):
    """Structural similarity of two H x W x 3 images in [0, 1], Gaussian window."""
    g = torch.exp(-(torch.arange(size, device=x.device) - size // 2) ** 2 / (2 * sigma**2))
    g = (g / g.sum())[None, None]
    win = (g.transpose(-1, -2) @ g).expand(3, 1, size, size)
    x, y = x.permute(2, 0, 1)[None], y.permute(2, 0, 1)[None]
    blur = lambda z: F.conv2d(z, win, groups=3)
    mx, my = blur(x), blur(y)
    vx, vy, cxy = blur(x * x) - mx**2, blur(y * y) - my**2, blur(x * y) - mx * my
    c1, c2 = 0.01**2, 0.03**2
    return (((2 * mx * my + c1) * (2 * cxy + c2)) / ((mx**2 + my**2 + c1) * (vx + vy + c2))).mean()


def render(splat, view, background=(0.0, 0.0, 0.0)):
    """Render a splat into a view. Returns the H x W x 3 image and the H x W opacity."""
    from gsplat import rasterization

    h, w = view["image"].shape[:2]
    dev = splat["means"].device
    colors, alphas, _ = rasterization(
        splat["means"], F.normalize(splat["quats"], dim=-1), torch.exp(splat["scales"]),
        torch.sigmoid(splat["opacities"]), splat["sh0"],
        torch.tensor(view["viewmat"], dtype=torch.float32, device=dev)[None],
        torch.tensor(view["K"], dtype=torch.float32, device=dev)[None], w, h, sh_degree=0,
        backgrounds=torch.tensor(background, dtype=torch.float32, device=dev)[None], packed=False)
    return colors[0].clamp(0, 1), alphas[0, ..., 0]


def quick_splat(views, points, colors, use_masks: bool, steps: int = 2000, seed: int = 0):
    """Train a 3D Gaussian splat for `steps` iterations on the given views.

    With use_masks, each target pixel outside the object's mask is replaced by a random
    background color, drawn anew every step and also used behind the splat, so only a
    transparent splat outside the object can match it. Without masks, every pixel of the
    photo is a target. Loss: 0.8 L1 + 0.2 (1 - SSIM), as in 3DGS.
    """
    from gsplat import rasterization
    from gsplat.strategy import DefaultStrategy

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    dev = "cuda"
    data = []
    for v in views:
        image = torch.tensor(v["image"], dtype=torch.float32, device=dev) / 255
        mask = torch.tensor(v["mask"], dtype=torch.float32, device=dev)[..., None] if use_masks else None
        data.append((image, mask, torch.tensor(v["viewmat"], dtype=torch.float32, device=dev)[None],
                     torch.tensor(v["K"], dtype=torch.float32, device=dev)[None]))
    h, w = views[0]["image"].shape[:2]

    # One Gaussian per sparse point, sized by the distance to its three nearest neighbors.
    d, _ = cKDTree(points).query(points, k=4)
    scale0 = np.log(np.maximum(d[:, 1:].mean(1), 1e-6))
    n = len(points)
    centers = np.array([-v["viewmat"][:3, :3].T @ v["viewmat"][:3, 3] for v in views])
    scene_scale = 1.1 * np.linalg.norm(centers - centers.mean(0), axis=1).max()
    splat = torch.nn.ParameterDict({
        "means": torch.nn.Parameter(torch.tensor(points, dtype=torch.float32)),
        "scales": torch.nn.Parameter(torch.tensor(scale0, dtype=torch.float32)[:, None].repeat(1, 3)),
        "quats": torch.nn.Parameter(torch.rand(n, 4)),
        "opacities": torch.nn.Parameter(torch.logit(torch.full((n,), 0.1))),
        "sh0": torch.nn.Parameter(((torch.tensor(colors, dtype=torch.float32) / 255 - 0.5) / 0.28209479)[:, None]),
    }).to(dev)
    lrs = {"means": 1.6e-4 * scene_scale, "scales": 5e-3, "quats": 1e-3, "opacities": 5e-2, "sh0": 2.5e-3}
    optimizers = {k: torch.optim.Adam([{"params": splat[k], "lr": lr, "name": k}], eps=1e-15)
                  for k, lr in lrs.items()}
    decay = torch.optim.lr_scheduler.ExponentialLR(optimizers["means"], gamma=0.01 ** (1 / steps))
    strategy = DefaultStrategy(verbose=False)  # densify from step 500 every 100 steps, as in 3DGS
    strategy.check_sanity(splat, optimizers)
    state = strategy.initialize_state(scene_scale=scene_scale)

    for step in range(steps):
        image, mask, viewmat, K = data[rng.integers(len(data))]
        background = torch.rand(3, device=dev) if use_masks else torch.zeros(3, device=dev)
        rendered, _, info = rasterization(
            splat["means"], F.normalize(splat["quats"], dim=-1), torch.exp(splat["scales"]),
            torch.sigmoid(splat["opacities"]), splat["sh0"], viewmat, K, w, h, sh_degree=0,
            backgrounds=background[None], packed=False)
        target = image if mask is None else image * mask + background * (1 - mask)
        loss = 0.8 * (rendered[0] - target).abs().mean() + 0.2 * (1 - _ssim(rendered[0], target))
        strategy.step_pre_backward(splat, optimizers, state, step, info)
        loss.backward()
        for opt in optimizers.values():
            opt.step()
            opt.zero_grad(set_to_none=True)
        decay.step()
        strategy.step_post_backward(splat, optimizers, state, step, info, packed=False)
    return {k: v.detach() for k, v in splat.items()}


def in_hull(splat, views, min_inside: float = 0.9):
    """True for each Gaussian whose center projects inside the mask in at least min_inside of
    the views that see it: the Gaussians inside the object's visual hull."""
    means = splat["means"]
    seen = torch.zeros(len(means), device=means.device)
    inside = torch.zeros_like(seen)
    for v in views:
        R = torch.tensor(v["viewmat"][:3, :3], dtype=torch.float32, device=means.device)
        t = torch.tensor(v["viewmat"][:3, 3], dtype=torch.float32, device=means.device)
        K = torch.tensor(v["K"], dtype=torch.float32, device=means.device)
        cam = means @ R.T + t
        uv = (cam @ K.T)[:, :2] / cam[:, 2:3].clamp(min=1e-6)
        h, w = v["mask"].shape
        x, y = uv[:, 0].long(), uv[:, 1].long()
        ok = (cam[:, 2] > 0) & (x >= 0) & (x < w) & (y >= 0) & (y < h)
        m = torch.tensor(v["mask"], device=means.device)
        seen += ok
        inside[ok] += m[y[ok], x[ok]].float()
    return (seen > 0) & (inside >= min_inside * seen)


def masked_psnr(splat, views, background=(1.0, 1.0, 1.0)) -> float:
    """Mean PSNR over the object's pixels (its mask) in the given views."""
    out = []
    for v in views:
        rendered, _ = render(splat, v, background)
        image = torch.tensor(v["image"], dtype=torch.float32, device=rendered.device) / 255
        m = torch.tensor(v["mask"], device=rendered.device)
        mse = ((rendered - image)[m] ** 2).mean()
        out.append(float(-10 * torch.log10(mse)))
    return float(np.mean(out))

In [ ]:
keep = [p for p in train if frames.index(p) not in flagged]  # frames with drifting masks stay out of both splats
train_views, points, point_colors = load_views(STAGE1, keep, masks, data_factor=2)
test_views, _, _ = load_views(STAGE1, [p for p in frames if p.name in heldout], masks, data_factor=2)
t = time.time()
render({"means": torch.zeros(1, 3, device="cuda"), "quats": torch.ones(1, 4, device="cuda"),
        "scales": torch.zeros(1, 3, device="cuda"), "opacities": torch.zeros(1, device="cuda"),
        "sh0": torch.zeros(1, 1, 3, device="cuda")}, train_views[0])  # the first call builds the GPU kernels
print(f"splatting kernels ready in {time.time() - t:.0f} s")
splats = {}
for name, use_masks in [("masked", True), ("unmasked", False)]:
    t = time.time()
    splats[name] = quick_splat(train_views, points, point_colors, use_masks=use_masks, steps=2000)
    print(f"{name} splat trained in {time.time() - t:.0f} s")
hull = in_hull(splats["unmasked"], train_views)
splats["unmasked, cut to the hull"] = {k: v[hull] for k, v in splats["unmasked"].items()}
stats = {}
for name, splat in splats.items():
    stats[name] = {"gaussians": len(splat["means"]),
                   "in_hull": float(in_hull(splat, train_views).float().mean()),
                   "heldout_psnr": masked_psnr(splat, test_views)}
    print(f"{name:>26}: {stats[name]['gaussians']:>9,} Gaussians, {100 * stats[name]['in_hull']:5.1f}% inside the "
          f"visual hull, held-out PSNR on the object {stats[name]['heldout_psnr']:.2f} dB")

PREVIEW_VIEW = test_views[0]
previews = {
    "masked": render(splats["masked"], PREVIEW_VIEW, background=(1.0, 1.0, 1.0))[0],
    "unmasked": render(splats["unmasked"], PREVIEW_VIEW)[0],
    "unmasked_cut": render(splats["unmasked, cut to the hull"], PREVIEW_VIEW, background=(1.0, 1.0, 1.0))[0],
}
previews = {k: (v.cpu().numpy() * 255).astype(np.uint8) for k, v in previews.items()}
print(f"previews rendered from the held-out frame {PREVIEW_VIEW['name']}")

In [ ]:
m = PREVIEW_VIEW["mask"]
ys, xs = np.nonzero(m)
pad = int(0.25 * max(ys.max() - ys.min(), xs.max() - xs.min()))
y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, m.shape[0])
x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, m.shape[1])
fig, axes = plt.subplots(2, 3, figsize=(15, 6.6), gridspec_kw={"height_ratios": [1, 1.3]})
panels = [("held-out photo " + PREVIEW_VIEW["name"], PREVIEW_VIEW["image"]),
          ("trained with masks", previews["masked"]),
          ("trained without masks", previews["unmasked"])]
for col, (title, img) in enumerate(panels):
    axes[0, col].imshow(img)
    axes[0, col].set_title(title, fontsize=10)
zoom = [("photo, close up", PREVIEW_VIEW["image"]), ("with masks, close up", previews["masked"]),
        ("without masks, cut to the visual hull", previews["unmasked_cut"])]
for col, (title, img) in enumerate(zoom):
    axes[1, col].imshow(img[y0:y1, x0:x1])
    axes[1, col].set_title(title, fontsize=10)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=80, pil_kwargs={"quality": 85})
plt.close(fig)
display(Image(data=buf.getvalue()))

## Frames and masks in the viewer

The viewer steps through the frames on the `frame` timeline, with the mask drawn over each frame as a segmentation image and the mask area plotted below it. A flagged frame shows up as a spike or a dip in that plot. The three preview renders are logged next to them.


In [ ]:
def log_stage2(frames, masks, areas, previews):
    rr.log("frames", rr.AnnotationContext([(0, "background", (0, 0, 0)), (1, "object", (255, 80, 0))]), static=True)
    for i, p in enumerate(frames):
        rr.set_time("frame", sequence=i)
        img = cv2.imread(str(p))
        k = 480 / img.shape[1]
        small = cv2.resize(img, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
        rr.log("frames/image", rr.EncodedImage(contents=cv2.imencode(".jpg", small)[1].tobytes(),
                                               media_type="image/jpeg"))
        mask = cv2.resize(masks[p.name].astype(np.uint8), (small.shape[1], small.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
        rr.log("frames/mask", rr.SegmentationImage(mask))
        rr.log("frames/mask_area", rr.Scalars(areas[i]))
    for name, img in previews.items():
        rr.log(f"preview/{name}", rr.Image(img), static=True)


log_stage2(frames, masks, areas, previews)

In [ ]:
show_rerun()

## Check

The check writes `stage2/` only when there is one mask for every stage 1 frame and not every mask is empty. On the reference object it also compares each mask with the matting mask that comes with the dataset, by intersection over union, and requires a mean IoU of at least 0.90. It lists the frames where the IoU falls below 0.5, because there one of the two masks is wrong, and the check cannot tell which one. On your own object there is no ground truth, so flagged frames, many flagged frames, and a masked splat below 20 dB on the held-out frames are warnings. A splat that poor usually means bad poses or a blurred capture, and stage 3 will not fix either.


In [ ]:
import json

import cv2
import numpy as np
from pathlib import Path


def write_stage2(out: Path, frames: list, masks: dict, areas: list, flagged: list, box, previews: dict) -> None:
    """Write stage2/ in one step: a mask for every frame, the previews and the drift record."""
    missing = [Path(p).name for p in frames if Path(p).name not in masks]
    if missing:
        raise RuntimeError(f"no mask for {len(missing)} frame(s), starting with {missing[0]}; stage2/ was not written")
    with atomic_dir(out) as d:
        (d / "masks").mkdir()
        for p in frames:
            name = Path(p).name
            cv2.imwrite(str(d / "masks" / (Path(name).stem + ".png")), masks[name].astype(np.uint8) * 255)
        (d / "preview").mkdir()
        for name, img in previews.items():
            cv2.imwrite(str(d / "preview" / f"{name}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        drift = {"box_xyxy": [int(v) for v in box], "prompt_frame": Path(frames[0]).name, "threshold": 0.3,
                 "areas": {Path(p).name: int(a) for p, a in zip(frames, areas)},
                 "flagged": [Path(frames[i]).name for i in flagged]}
        (d / "drift.json").write_text(json.dumps(drift, indent=2))

In [ ]:
import cv2
import numpy as np
from pathlib import Path


def reference_iou(masks: dict, matting: Path) -> dict:
    """IoU of each mask against the dataset's matting mask of the same frame."""
    out = {}
    for name, m in masks.items():
        gt = cv2.imread(str(Path(matting) / (Path(name).stem + ".png")), cv2.IMREAD_GRAYSCALE) > 127
        out[name] = float((m & gt).sum() / max((m | gt).sum(), 1))
    return out


IOU_GATE_OBJECT = "boxed_beverage_001"  # the reference object the 0.90 IoU gate was calibrated on


def stage2_problems(frames: list, masks: dict, flagged: list, stats: dict, iou: dict | None,
                    gate_iou: bool = True):
    """Return (failures, warnings) for the stage 2 check. iou is None off the reference object.

    gate_iou: the mean IoU below 0.90 fails only on the reference object the gate was set on
    (IOU_GATE_OBJECT); on another reference object it is a warning.
    """
    failures, warnings = [], []
    names = [Path(p).name for p in frames]
    if sorted(masks) != sorted(names):
        failures.append("the masks do not cover exactly the stage 1 frames")
        return failures, warnings
    empty = [n for n in names if not masks[n].any()]
    if len(empty) == len(names):
        failures.append("every mask is empty: check the box prompt")
    if flagged:
        warnings.append(f"{len(flagged)} frame(s) changed area by more than 30% against their neighbors: "
                        + ", ".join(names[i] for i in flagged) + ". They are kept out of both quick splats; "
                        "look at them in the viewer, and add a second box prompt or recapture if they are wrong")
    if len(flagged) > 0.1 * len(names):
        warnings.append("more than 10% of the frames drift: the object may be too shiny, too dark or too "
                        "small in the frame for the tracker")
    if stats["masked"]["heldout_psnr"] < 20:
        warnings.append(f"the masked quick splat reaches only {stats['masked']['heldout_psnr']:.1f} dB on the "
                        "held-out frames: check the stage 1 poses and the capture before stage 3")
    if iou is not None:
        mean = float(np.mean(list(iou.values())))
        if mean < 0.90:
            message = f"mean IoU against the reference matting is {mean:.3f} (want at least 0.90)"
            if gate_iou:
                failures.append(message)
            else:
                warnings.append(message + f"; the gate was set on {IOU_GATE_OBJECT}, so here it is printed only")
    return failures, warnings

In [ ]:
iou = reference_iou(masks, PROJECT / "ground_truth" / "matting") if REFERENCE else None
print(f"{len(masks)} masks for {len(frames)} frames, {len(flagged)} flagged; stage time {(time.time() - T0) / 60:.1f} min")
if iou is not None:
    values = np.array([iou[p.name] for p in frames])
    worst = frames[int(values.argmin())].name
    print(f"mean IoU against the OmniObject3D matting {values.mean():.3f} over {len(values)} frames, "
          f"minimum {values.min():.3f} ({worst})")
    low = [p.name for p in frames if iou[p.name] < 0.5]
    if low:
        rest = values[values >= 0.5]
        print(f"IoU below 0.5 on {len(low)} frame(s): {', '.join(low)}; "
              f"on the other {len(rest)} the mean is {rest.mean():.3f} and the minimum {rest.min():.3f}")
failures, warnings = stage2_problems(frames, masks, flagged, stats, iou,
                                     gate_iou=bool(SCAN_OBJECT) and SCAN_OBJECT.endswith(IOU_GATE_OBJECT))
for w in warnings:
    print("warning:", w)
assert not failures, "; ".join(failures)  # nothing is written when a gate fails
write_stage2(PROJECT / "stage2", frames, masks, areas, flagged, box, previews)
if REFERENCE:
    (PROJECT / "stage2" / ".reference").touch()  # stage 3 then knows it runs on the reference
print("passed")

### Question 1. What SAM 2 carries forward

Using the memory bank above, explain how SAM 2 computes the mask of a frame far from the prompted one, and how a single wrong mask can carry forward through the bank. Point to a frame in the viewer where the object looks most unlike your prompted frame.

*Write your answer here.*

### Question 2. What the drift check sees

Describe one kind of wrong mask that the area rule cannot detect, and explain how the poses of stage 1 could catch it. Point to the largest change between consecutive frames that your run printed, or to a flagged frame.

*Write your answer here.*

### Question 3. Training a splat with masks

Using the blending equation, explain why replacing the background with a random color at every step empties the splat outside the object. Then explain why cutting a splat trained without masks down to the visual hull does not give the same result. Point to the row of the splat table, or the part of the preview, that shows the difference.

*Write your answer here.*